# Typewell Alignment Feature Screen

Research-guided alignment screen for the Bayesian/geologic modeling track. This notebook asks whether fold-safe typewell, analog-well, and pseudo-zone descriptors explain residuals left by the current Huber + additive Hilbert residual model.

## 1. Setup

In [1]:
"Configure imports, paths, run metadata, and feature-screen constants."

from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, RobustScaler, StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "context").exists():
    PROJECT_ROOT = next(parent for parent in PROJECT_ROOT.parents if (parent / "context").exists())

DATE_STAMP = "2026-06-23"
RUN_ID = f"{DATE_STAMP}__typewell_alignment_feature_screen"

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "train"
ARTIFACT_TABLES = PROJECT_ROOT / "artifacts" / "tables"
ARTIFACT_REPORTS = PROJECT_ROOT / "artifacts" / "reports"
ARTIFACT_LOGS = PROJECT_ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for path in [ARTIFACT_TABLES, ARTIFACT_REPORTS, ARTIFACT_LOGS]:
    path.mkdir(parents=True, exist_ok=True)

RESIDUAL_PATH = ARTIFACT_TABLES / "model__hsgp_residual_on_gam_huber_residual_sample__2026-06-22.csv"
TARGET_CANDIDATE = "huber_plus_hsgp_dps_m8"
LOCAL_REFERENCE_RMSE = 14.3586
PUBLIC_REFERENCE_RMSE = 14.514
OFFSET_CLIP = 12.0
RANDOM_SEED = 20260623

ROW_FEATURES = ["eval_progress", "distance_log1p", "abs_projected_delta_10"]
ALIGNMENT_FEATURES = [
    "known_rows",
    "known_tvt_span",
    "known_md_span",
    "known_gr_mean_last300",
    "known_gr_std_last300",
    "known_gr_slope_last300",
    "typewell_corr_last300",
    "typewell_mae_last300",
    "typewell_best_lag_corr",
    "typewell_best_lag_tvt",
    "typewell_best_lag_mae",
    "xy_span_known",
    "z_drop_known",
    "last_known_tvt",
]
ANALOG_FEATURES = [
    "analog_distance",
    "analog_delta_pred",
    "analog_delta_minus_baseline_delta",
    "analog_last_tvt_diff",
    "analog_typewell_corr_diff",
]

print(f"project_root={PROJECT_ROOT}")
print(f"run_id={RUN_ID}")

project_root=C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
run_id=2026-06-23__typewell_alignment_feature_screen


## 2. Load Residual Sample

The screen uses the same out-of-fold residual rows from the current Huber + additive Hilbert model. Every correction arm trains on other folds and scores the held-out fold.

In [2]:
"Load the current best residual sample and define the same-sample baseline."

residuals_all = pd.read_csv(RESIDUAL_PATH)
residuals = residuals_all.loc[residuals_all["candidate"].eq(TARGET_CANDIDATE)].copy()
for col in ["fold", "row_idx", "distance_from_boundary_md", "eval_progress", "abs_projected_delta_10", "y_true", "y_pred", "residual"]:
    residuals[col] = pd.to_numeric(residuals[col], errors="coerce")
residuals["fold"] = residuals["fold"].astype(int)
residuals["row_idx"] = residuals["row_idx"].astype(int)
residuals["distance_log1p"] = np.log1p(residuals["distance_from_boundary_md"].clip(lower=0))
residuals["baseline_pred"] = residuals["y_pred"]
residuals["baseline_residual"] = residuals["y_true"] - residuals["baseline_pred"]

print(f"rows={len(residuals):,}")
print(f"wells={residuals['wellname'].nunique():,}")
print(residuals.groupby("fold")["wellname"].nunique().rename("wells_by_fold").to_string())

rows=5,400
wells=773
fold
1    258
2    258
3    257


## 3. Typewell And Known-Zone Descriptors

These descriptors use only known-zone information from each horizontal well plus the provided typewell curve. The lag features are a light DTW-style proxy: shift the typewell TVT coordinate and keep the best known-zone GR agreement.

In [3]:
"Build per-well known-zone, typewell-alignment, and trajectory descriptors."

LAGS = np.arange(-40.0, 42.5, 2.5)


def safe_corr(a, b):
    left = np.asarray(a, dtype=float)
    right = np.asarray(b, dtype=float)
    mask = np.isfinite(left) & np.isfinite(right)
    if mask.sum() < 5:
        return np.nan
    if np.nanstd(left[mask]) < 1e-9 or np.nanstd(right[mask]) < 1e-9:
        return np.nan
    return float(np.corrcoef(left[mask], right[mask])[0, 1])


def linear_slope(x, y):
    left = np.asarray(x, dtype=float)
    right = np.asarray(y, dtype=float)
    mask = np.isfinite(left) & np.isfinite(right)
    if mask.sum() < 5:
        return np.nan
    coef = np.polyfit(left[mask], right[mask], 1)
    return float(coef[0])


def interp_typewell_gr(typewell, tvt_values, lag=0.0):
    clean = typewell[["TVT", "GR"]].dropna().sort_values("TVT")
    if len(clean) < 5:
        return np.full(len(tvt_values), np.nan)
    return np.interp(np.asarray(tvt_values, dtype=float) + lag, clean["TVT"].to_numpy(), clean["GR"].to_numpy(), left=np.nan, right=np.nan)


def load_well_frames(wellname):
    horizontal = pd.read_csv(DATA_DIR / f"{wellname}__horizontal_well.csv")
    typewell = pd.read_csv(DATA_DIR / f"{wellname}__typewell.csv")
    return horizontal, typewell


def build_well_descriptor(wellname):
    horizontal, typewell = load_well_frames(wellname)
    known = horizontal.loc[horizontal["TVT_input"].notna()].copy()
    tail = known.tail(300).copy()
    descriptor = {"wellname": wellname}
    descriptor["known_rows"] = float(len(known))
    descriptor["known_tvt_span"] = float(known["TVT_input"].max() - known["TVT_input"].min()) if len(known) else np.nan
    descriptor["known_md_span"] = float(known["MD"].max() - known["MD"].min()) if len(known) else np.nan
    descriptor["known_gr_mean_last300"] = float(tail["GR"].mean()) if len(tail) else np.nan
    descriptor["known_gr_std_last300"] = float(tail["GR"].std()) if len(tail) else np.nan
    descriptor["known_gr_slope_last300"] = linear_slope(tail["TVT_input"], tail["GR"])
    descriptor["xy_span_known"] = float(np.hypot(known["X"].max() - known["X"].min(), known["Y"].max() - known["Y"].min())) if len(known) else np.nan
    descriptor["z_drop_known"] = float(known["Z"].iloc[-1] - known["Z"].iloc[0]) if len(known) > 1 else np.nan
    descriptor["last_known_tvt"] = float(known["TVT_input"].iloc[-1]) if len(known) else np.nan
    if len(tail) >= 5:
        baseline_gr = interp_typewell_gr(typewell, tail["TVT_input"].to_numpy(), lag=0.0)
        descriptor["typewell_corr_last300"] = safe_corr(tail["GR"], baseline_gr)
        descriptor["typewell_mae_last300"] = float(np.nanmean(np.abs(tail["GR"].to_numpy() - baseline_gr)))
        lag_rows = []
        for lag in LAGS:
            shifted_gr = interp_typewell_gr(typewell, tail["TVT_input"].to_numpy(), lag=lag)
            corr = safe_corr(tail["GR"], shifted_gr)
            mae = float(np.nanmean(np.abs(tail["GR"].to_numpy() - shifted_gr))) if np.isfinite(shifted_gr).sum() >= 5 else np.nan
            lag_rows.append((lag, corr, mae))
        lag_frame = pd.DataFrame(lag_rows, columns=["lag", "corr", "mae"])
        lag_frame["rank_corr"] = lag_frame["corr"].fillna(-999.0)
        best = lag_frame.sort_values(["rank_corr", "mae"], ascending=[False, True]).iloc[0]
        descriptor["typewell_best_lag_corr"] = float(best["corr"])
        descriptor["typewell_best_lag_tvt"] = float(best["lag"])
        descriptor["typewell_best_lag_mae"] = float(best["mae"])
    else:
        descriptor["typewell_corr_last300"] = np.nan
        descriptor["typewell_mae_last300"] = np.nan
        descriptor["typewell_best_lag_corr"] = np.nan
        descriptor["typewell_best_lag_tvt"] = np.nan
        descriptor["typewell_best_lag_mae"] = np.nan
    return descriptor

well_descriptors = pd.DataFrame([build_well_descriptor(well) for well in sorted(residuals["wellname"].unique())])
print(well_descriptors.head().to_string(index=False))
print(well_descriptors[ALIGNMENT_FEATURES].describe().T[["mean", "std", "min", "max"]].round(3).to_string())

wellname  known_rows  known_tvt_span  known_md_span  known_gr_mean_last300  known_gr_std_last300  known_gr_slope_last300  xy_span_known  z_drop_known  last_known_tvt  typewell_corr_last300  typewell_mae_last300  typewell_best_lag_corr  typewell_best_lag_tvt  typewell_best_lag_mae
000d7d20      1442.0          520.10         1441.0             106.326237              6.068260                3.165573    1190.815381       -476.51        11747.37              -0.398691             11.532797                0.553606                  -22.5              19.894136
00bbac68      1545.0          818.42         1544.0              87.895091             10.138084                6.181385    1077.617096       -736.48        12223.54               0.096803             10.819063                0.773944                  -15.0               8.951110
00e12e8b      2083.0          998.64         2082.0              32.240742              6.211680               -0.113335    1453.191016       -968.90        

## 4. Fold-Safe Analog Features

For each held-out fold, analog templates come only from the other folds. The analog TVT-delta curve is interpolated by evaluation progress and compared with the baseline prediction delta.

In [4]:
"Create fold-safe nearest-analog transfer features without using validation targets."

model_frame = residuals.merge(well_descriptors, on="wellname", how="left")
model_frame["baseline_delta_from_last_known"] = model_frame["baseline_pred"] - model_frame["last_known_tvt"]
model_frame["true_delta_from_last_known"] = model_frame["y_true"] - model_frame["last_known_tvt"]

ANALOG_DESCRIPTOR_FEATURES = [
    "known_tvt_span",
    "known_md_span",
    "known_gr_mean_last300",
    "known_gr_std_last300",
    "known_gr_slope_last300",
    "typewell_corr_last300",
    "typewell_best_lag_corr",
    "typewell_best_lag_tvt",
    "xy_span_known",
    "z_drop_known",
]


def median_impute_scale(train_desc, all_desc, feature_cols):
    med = train_desc[feature_cols].median(numeric_only=True)
    scale = train_desc[feature_cols].std(numeric_only=True).replace(0, np.nan).fillna(1.0)
    train_scaled = (train_desc[feature_cols].fillna(med) - med) / scale
    all_scaled = (all_desc[feature_cols].fillna(med) - med) / scale
    return train_scaled, all_scaled


def interpolate_analog_delta(train_rows, analog_well, progress_values):
    curve = train_rows.loc[train_rows["wellname"].eq(analog_well), ["eval_progress", "true_delta_from_last_known"]].dropna().sort_values("eval_progress")
    if len(curve) < 2:
        return np.full(len(progress_values), np.nan)
    grouped = curve.groupby("eval_progress", as_index=False)["true_delta_from_last_known"].mean()
    return np.interp(np.asarray(progress_values, dtype=float), grouped["eval_progress"].to_numpy(), grouped["true_delta_from_last_known"].to_numpy(), left=grouped["true_delta_from_last_known"].iloc[0], right=grouped["true_delta_from_last_known"].iloc[-1])

analog_chunks = []
for fold in sorted(model_frame["fold"].unique()):
    train_rows = model_frame.loc[~model_frame["fold"].eq(fold)].copy()
    valid_rows = model_frame.loc[model_frame["fold"].eq(fold)].copy()
    train_desc = well_descriptors.loc[well_descriptors["wellname"].isin(train_rows["wellname"].unique())].copy()
    valid_desc = well_descriptors.loc[well_descriptors["wellname"].isin(valid_rows["wellname"].unique())].copy()
    train_scaled, all_scaled = median_impute_scale(train_desc, pd.concat([train_desc, valid_desc], ignore_index=True), ANALOG_DESCRIPTOR_FEATURES)
    train_scaled.index = train_desc["wellname"].to_numpy()
    all_scaled.index = pd.concat([train_desc, valid_desc], ignore_index=True)["wellname"].to_numpy()
    train_matrix = train_scaled.to_numpy(dtype=float)
    train_names = train_scaled.index.to_numpy()
    fold_rows = []
    for source in [train_rows, valid_rows]:
        source_kind = "train" if source is train_rows else "valid"
        for wellname, rows in source.groupby("wellname"):
            query = all_scaled.loc[wellname].to_numpy(dtype=float)
            distances = np.sqrt(np.mean((train_matrix - query) ** 2, axis=1))
            if source_kind == "train":
                distances = distances.copy()
                distances[train_names == wellname] = np.inf
            nearest_pos = int(np.nanargmin(distances))
            analog_well = str(train_names[nearest_pos])
            analog_delta = interpolate_analog_delta(train_rows, analog_well, rows["eval_progress"].to_numpy())
            out = rows[["fold", "wellname", "row_idx", "eval_progress", "baseline_delta_from_last_known"]].copy()
            out["analog_well"] = analog_well
            out["analog_distance"] = float(distances[nearest_pos])
            out["analog_delta_pred"] = analog_delta
            out["analog_delta_minus_baseline_delta"] = out["analog_delta_pred"] - out["baseline_delta_from_last_known"]
            out["analog_last_tvt_diff"] = float(well_descriptors.loc[well_descriptors["wellname"].eq(wellname), "last_known_tvt"].iloc[0] - well_descriptors.loc[well_descriptors["wellname"].eq(analog_well), "last_known_tvt"].iloc[0])
            out["analog_typewell_corr_diff"] = float(well_descriptors.loc[well_descriptors["wellname"].eq(wellname), "typewell_corr_last300"].iloc[0] - well_descriptors.loc[well_descriptors["wellname"].eq(analog_well), "typewell_corr_last300"].iloc[0])
            fold_rows.append(out)
    analog_chunks.append(pd.concat(fold_rows, ignore_index=True))

analog_features = pd.concat(analog_chunks, ignore_index=True).drop_duplicates(["fold", "wellname", "row_idx"], keep="last")
model_frame = model_frame.merge(analog_features[["fold", "wellname", "row_idx", "analog_well"] + ANALOG_FEATURES], on=["fold", "wellname", "row_idx"], how="left")
print(model_frame[["wellname", "row_idx", "analog_well"] + ANALOG_FEATURES].head().to_string(index=False))
print(model_frame[ANALOG_FEATURES].describe().T[["mean", "std", "min", "max"]].round(3).to_string())

wellname  row_idx analog_well  analog_distance  analog_delta_pred  analog_delta_minus_baseline_delta  analog_last_tvt_diff  analog_typewell_corr_diff
000d7d20     1442    3f603129         0.585683           2.390000                           5.418650                773.09                  -0.318155
000d7d20     1993    3f603129         0.585683          19.572190                          21.817007                773.09                  -0.318155
000d7d20     2545    3f603129         0.585683          41.535435                          43.971705                773.09                  -0.318155
000d7d20     3097    3f603129         0.585683          37.538082                          39.039547                773.09                  -0.318155
000d7d20     3649    3f603129         0.585683          19.884118                          21.099572                773.09                  -0.318155
                                     mean      std       min       max
analog_distance              

## 5. Candidate Arms

In [5]:
"Define conservative residual-correction arms for alignment and analog features."

BASE_FEATURES = ROW_FEATURES
MAIN_ALIGNMENT_FEATURES = ALIGNMENT_FEATURES + ANALOG_FEATURES
ROW_ALIGNMENT_FEATURES = ROW_FEATURES + ALIGNMENT_FEATURES + ANALOG_FEATURES

CANDIDATES = [
    {"name": "baseline_no_alignment", "kind": "baseline", "alpha": 0.0, "shrink": 0.0, "prior": "zero residual correction"},
    {"name": "analog_direct_s005", "kind": "analog_direct", "alpha": 0.0, "shrink": 0.05, "prior": "five percent shrink toward nearest analog transferred delta"},
    {"name": "analog_direct_s010", "kind": "analog_direct", "alpha": 0.0, "shrink": 0.10, "prior": "ten percent shrink toward nearest analog transferred delta"},
    {"name": "alignment_main_ridge_a500_s025", "kind": "alignment_main", "alpha": 500.0, "shrink": 0.25, "prior": "alignment descriptors with strong ridge and 75 percent shrinkage"},
    {"name": "alignment_main_ridge_a1500_s025", "kind": "alignment_main", "alpha": 1500.0, "shrink": 0.25, "prior": "alignment descriptors with very strong ridge and 75 percent shrinkage"},
    {"name": "alignment_row_ridge_a1000_s025", "kind": "alignment_row", "alpha": 1000.0, "shrink": 0.25, "prior": "row plus alignment descriptors with strong ridge"},
    {"name": "alignment_row_poly2_a2500_s015", "kind": "alignment_poly2", "alpha": 2500.0, "shrink": 0.15, "prior": "small quadratic alignment surface with heavy shrinkage"},
    {"name": "alignment_row_huber_a010_s025", "kind": "alignment_huber", "alpha": 0.10, "shrink": 0.25, "prior": "robust alignment residual correction"},
]


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def clipped(values):
    return np.clip(np.asarray(values, dtype=float), -OFFSET_CLIP, OFFSET_CLIP)


def add_alignment_interactions(frame):
    out = frame.copy()
    selected = ["typewell_best_lag_corr", "typewell_best_lag_tvt", "analog_delta_minus_baseline_delta", "analog_distance", "known_gr_slope_last300"]
    for align_col in selected:
        for row_col in ROW_FEATURES:
            out[f"{align_col}__x__{row_col}"] = out[align_col] * out[row_col]
    return out

INTERACTION_FEATURES = ROW_ALIGNMENT_FEATURES + [f"{a}__x__{r}" for a in ["typewell_best_lag_corr", "typewell_best_lag_tvt", "analog_delta_minus_baseline_delta", "analog_distance", "known_gr_slope_last300"] for r in ROW_FEATURES]

print(pd.DataFrame(CANDIDATES)[["name", "kind", "alpha", "shrink", "prior"]].to_string(index=False))

                           name            kind  alpha  shrink                                                                 prior
          baseline_no_alignment        baseline    0.0    0.00                                              zero residual correction
             analog_direct_s005   analog_direct    0.0    0.05           five percent shrink toward nearest analog transferred delta
             analog_direct_s010   analog_direct    0.0    0.10            ten percent shrink toward nearest analog transferred delta
 alignment_main_ridge_a500_s025  alignment_main  500.0    0.25      alignment descriptors with strong ridge and 75 percent shrinkage
alignment_main_ridge_a1500_s025  alignment_main 1500.0    0.25 alignment descriptors with very strong ridge and 75 percent shrinkage
 alignment_row_ridge_a1000_s025   alignment_row 1000.0    0.25                      row plus alignment descriptors with strong ridge
 alignment_row_poly2_a2500_s015 alignment_poly2 2500.0    0.15       

## 6. Out-Of-Fold Evaluation

In [6]:
"Train alignment corrections on other folds and score each held-out fold."

start_time = time.time()
predicted_rows = []
fold_metric_rows = []
for candidate in CANDIDATES:
    for fold in sorted(model_frame["fold"].unique()):
        train_rows = model_frame.loc[~model_frame["fold"].eq(fold)].copy()
        valid_rows = model_frame.loc[model_frame["fold"].eq(fold)].copy()
        raw_correction = np.zeros(len(valid_rows), dtype=float)
        if candidate["kind"] == "analog_direct":
            raw_correction = valid_rows["analog_delta_minus_baseline_delta"].fillna(0.0).to_numpy()
        elif candidate["kind"] == "alignment_main":
            model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", RobustScaler()), ("model", Ridge(alpha=candidate["alpha"]))])
            model.fit(train_rows[MAIN_ALIGNMENT_FEATURES], train_rows["baseline_residual"])
            raw_correction = model.predict(valid_rows[MAIN_ALIGNMENT_FEATURES])
        elif candidate["kind"] == "alignment_row":
            model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", Ridge(alpha=candidate["alpha"]))])
            model.fit(train_rows[ROW_ALIGNMENT_FEATURES], train_rows["baseline_residual"])
            raw_correction = model.predict(valid_rows[ROW_ALIGNMENT_FEATURES])
        elif candidate["kind"] == "alignment_poly2":
            model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("poly", PolynomialFeatures(degree=2, include_bias=False)), ("model", Ridge(alpha=candidate["alpha"]))])
            model.fit(train_rows[ROW_ALIGNMENT_FEATURES], train_rows["baseline_residual"])
            raw_correction = model.predict(valid_rows[ROW_ALIGNMENT_FEATURES])
        elif candidate["kind"] == "alignment_huber":
            train_x = add_alignment_interactions(train_rows)
            valid_x = add_alignment_interactions(valid_rows)
            model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", HuberRegressor(alpha=candidate["alpha"], epsilon=1.35, max_iter=500))])
            model.fit(train_x[INTERACTION_FEATURES], train_x["baseline_residual"])
            raw_correction = model.predict(valid_x[INTERACTION_FEATURES])
        correction = clipped(raw_correction) * candidate["shrink"]
        y_pred = valid_rows["baseline_pred"].to_numpy() + correction
        y_true = valid_rows["y_true"].to_numpy()
        fold_metric_rows.append(
            {
                "candidate": candidate["name"],
                "fold": int(fold),
                "rmse": rmse(y_true, y_pred),
                "mae": float(mean_absolute_error(y_true, y_pred)),
                "n_rows": int(len(valid_rows)),
                "n_wells": int(valid_rows["wellname"].nunique()),
                "mean_correction": float(np.mean(correction)),
                "std_correction": float(np.std(correction)),
                "max_abs_correction": float(np.max(np.abs(correction))) if len(correction) else 0.0,
            }
        )
        out = valid_rows[["fold", "wellname", "row_idx", "y_true", "baseline_pred", "baseline_residual", "eval_progress", "distance_from_boundary_md", "abs_projected_delta_10", "analog_well"]].copy()
        out["candidate"] = candidate["name"]
        out["alignment_correction"] = correction
        out["y_pred"] = y_pred
        out["residual_after_alignment"] = out["y_true"] - out["y_pred"]
        predicted_rows.append(out)

fold_metrics = pd.DataFrame(fold_metric_rows)
predictions = pd.concat(predicted_rows, ignore_index=True)
summary = (
    fold_metrics.groupby("candidate", as_index=False)
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        mean_correction=("mean_correction", "mean"),
        std_correction=("std_correction", "mean"),
        max_abs_correction=("max_abs_correction", "max"),
        n_rows=("n_rows", "sum"),
        n_wells=("n_wells", "sum"),
    )
)
baseline_rmse = float(summary.loc[summary["candidate"].eq("baseline_no_alignment"), "rmse_mean"].iloc[0])
summary["rmse_delta_vs_sample_baseline"] = summary["rmse_mean"] - baseline_rmse
summary = summary.sort_values(["rmse_mean", "candidate"]).reset_index(drop=True)
train_time_s = time.time() - start_time
print(summary.to_string(index=False))

                      candidate  rmse_mean  rmse_std  mae_mean  mean_correction  std_correction  max_abs_correction  n_rows  n_wells  rmse_delta_vs_sample_baseline
             analog_direct_s010  14.696599  0.426299 10.382586        -0.000565        0.901826            1.200000    5400      773                      -0.041946
             analog_direct_s005  14.710686  0.456576 10.388338        -0.000283        0.450913            0.600000    5400      773                      -0.027859
alignment_main_ridge_a1500_s025  14.733321  0.475644 10.417378         0.046060        0.394757            1.770606    5400      773                      -0.005224
 alignment_row_poly2_a2500_s015  14.736085  0.476527 10.425007         0.013649        0.548797            1.800000    5400      773                      -0.002460
 alignment_main_ridge_a500_s025  14.736379  0.466559 10.420553         0.037190        0.484711            2.246668    5400      773                      -0.002166
 alignment_row_r

## 7. Diagnostics And Decision

In [7]:
"Summarize the best alignment arm and residual diagnostics."

best_candidate = summary.iloc[0]["candidate"]
best_row = summary.iloc[0]
baseline_row = summary.loc[summary["candidate"].eq("baseline_no_alignment")].iloc[0]
best_predictions = predictions.loc[predictions["candidate"].eq(best_candidate)].copy()

well_after = (
    best_predictions.groupby(["fold", "wellname"], as_index=False)
    .agg(
        n_rows=("residual_after_alignment", "size"),
        mean_residual_after=("residual_after_alignment", "mean"),
        mae_after=("residual_after_alignment", lambda s: float(np.mean(np.abs(s)))),
        rmse_after=("residual_after_alignment", lambda s: float(np.sqrt(np.mean(np.asarray(s) ** 2)))),
        mean_alignment_correction=("alignment_correction", "mean"),
        alignment_correction_std=("alignment_correction", "std"),
    )
)
well_before = (
    model_frame.groupby(["fold", "wellname"], as_index=False)
    .agg(
        mean_residual_before=("baseline_residual", "mean"),
        rmse_before=("baseline_residual", lambda s: float(np.sqrt(np.mean(np.asarray(s) ** 2)))),
    )
)
well_diagnostics = well_after.merge(well_before, on=["fold", "wellname"], how="left")
well_diagnostics["abs_mean_residual_before"] = well_diagnostics["mean_residual_before"].abs()
well_diagnostics["abs_mean_residual_after"] = well_diagnostics["mean_residual_after"].abs()
well_diagnostics["abs_bias_delta"] = well_diagnostics["abs_mean_residual_after"] - well_diagnostics["abs_mean_residual_before"]

feature_correlations = []
for col in ALIGNMENT_FEATURES + ANALOG_FEATURES:
    corr = safe_corr(model_frame[col], model_frame["baseline_residual"])
    feature_correlations.append({"feature": col, "corr_with_baseline_residual": corr})
feature_correlations = pd.DataFrame(feature_correlations).sort_values("corr_with_baseline_residual", key=lambda s: s.abs(), ascending=False)

findings = pd.DataFrame(
    [
        {
            "finding": "best_candidate",
            "value": f"{best_candidate} RMSE {best_row['rmse_mean']:.4f} vs sample baseline {baseline_row['rmse_mean']:.4f}",
            "modeling_implication": "Promote only if alignment features clear the 0.05 RMSE feature-screen gate.",
        },
        {
            "finding": "sample_baseline_delta",
            "value": f"best candidate delta {best_row['rmse_delta_vs_sample_baseline']:.4f}",
            "modeling_implication": "Small deltas mean alignment should be redesigned before Bayesian hierarchy consumes it.",
        },
        {
            "finding": "well_bias_change",
            "value": f"median absolute well-bias delta {well_diagnostics['abs_bias_delta'].median():.4f}",
            "modeling_implication": "Negative is good; positive means alignment corrections add noise at the well level.",
        },
        {
            "finding": "strongest_alignment_correlation",
            "value": f"{feature_correlations.iloc[0]['feature']} corr {feature_correlations.iloc[0]['corr_with_baseline_residual']:.4f}",
            "modeling_implication": "Use this as the first clue for the next alignment-feature redesign.",
        },
    ]
)

print(findings.to_string(index=False))
print(feature_correlations.head(10).to_string(index=False))

                        finding                                                      value                                                                    modeling_implication
                 best_candidate analog_direct_s010 RMSE 14.6966 vs sample baseline 14.7385             Promote only if alignment features clear the 0.05 RMSE feature-screen gate.
          sample_baseline_delta                               best candidate delta -0.0419 Small deltas mean alignment should be redesigned before Bayesian hierarchy consumes it.
               well_bias_change                    median absolute well-bias delta -0.0988     Negative is good; positive means alignment corrections add noise at the well level.
strongest_alignment_correlation              analog_delta_minus_baseline_delta corr 0.0680                     Use this as the first clue for the next alignment-feature redesign.
                          feature  corr_with_baseline_residual
analog_delta_minus_baseline_delta         

## 8. Export Artifacts

In [8]:
"Export tables, report, run bundle, and model-comparison row."

summary_path = ARTIFACT_TABLES / f"model__typewell_alignment_feature_summary__{DATE_STAMP}.csv"
fold_path = ARTIFACT_TABLES / f"model__typewell_alignment_feature_fold_metrics__{DATE_STAMP}.csv"
pred_path = ARTIFACT_TABLES / f"model__typewell_alignment_feature_residual_sample__{DATE_STAMP}.csv"
well_path = ARTIFACT_TABLES / f"model__typewell_alignment_feature_well_diagnostics__{DATE_STAMP}.csv"
descriptor_path = ARTIFACT_TABLES / f"model__typewell_alignment_feature_descriptors__{DATE_STAMP}.csv"
correlation_path = ARTIFACT_TABLES / f"model__typewell_alignment_feature_correlations__{DATE_STAMP}.csv"
findings_path = ARTIFACT_TABLES / f"model__typewell_alignment_feature_findings__{DATE_STAMP}.csv"
report_path = ARTIFACT_REPORTS / f"model__typewell_alignment_feature_screen__{DATE_STAMP}.md"
comparison_path = ARTIFACT_TABLES / "model_comparison.csv"

summary.to_csv(summary_path, index=False)
fold_metrics.to_csv(fold_path, index=False)
predictions.to_csv(pred_path, index=False)
well_diagnostics.to_csv(well_path, index=False)
well_descriptors.to_csv(descriptor_path, index=False)
feature_correlations.to_csv(correlation_path, index=False)
findings.to_csv(findings_path, index=False)

report = f"""# Typewell Alignment Feature Screen

Run ID: `{RUN_ID}`

Result

Best candidate: `{best_candidate}`

- Same-sample baseline RMSE: **{baseline_row['rmse_mean']:.4f}**
- Best RMSE: **{best_row['rmse_mean']:.4f}**
- Delta vs same-sample baseline: **{best_row['rmse_delta_vs_sample_baseline']:.4f}**
- Current full local reference: **{LOCAL_REFERENCE_RMSE:.4f}**
- Current public reference: **{PUBLIC_REFERENCE_RMSE:.3f}**

Interpretation

This screen tests research-guided alignment features as residual corrections on the bounded Huber + additive Hilbert residual sample. The result should be interpreted against the same-sample no-alignment baseline, not directly against the full local diagnostic reference.

Decision

Promote only if the best alignment arm clears the **0.05 RMSE** screen gate and does not worsen fold-level diagnostics. If it fails, use the feature correlations and analog diagnostics to redesign alignment rather than stacking more residual models.
"""
report_path.write_text(report, encoding="utf-8")

comparison_row = pd.DataFrame(
    [
        {
            "run_id": RUN_ID,
            "model_family": "alignment_feature_screen",
            "model_name": best_candidate,
            "cv_rmse_mean": float(best_row["rmse_mean"]),
            "cv_rmse_std": float(best_row["rmse_std"]),
            "train_time_s": float(train_time_s),
            "n_features": int(len(ROW_ALIGNMENT_FEATURES)),
            "notes": "Fold-safe typewell/analog alignment residual-correction screen on Huber + additive Hilbert residual sample",
        }
    ]
)
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    comparison = comparison.loc[~comparison["run_id"].eq(RUN_ID)]
    comparison = pd.concat([comparison, comparison_row], ignore_index=True)
else:
    comparison = comparison_row
comparison.to_csv(comparison_path, index=False)

for src in [summary_path, fold_path, pred_path, well_path, descriptor_path, correlation_path, findings_path, report_path, comparison_path]:
    shutil.copy2(src, ARTIFACT_LOGS / src.name)

config = {
    "run_id": RUN_ID,
    "target_candidate": TARGET_CANDIDATE,
    "offset_clip": OFFSET_CLIP,
    "candidates": CANDIDATES,
    "row_features": ROW_FEATURES,
    "alignment_features": ALIGNMENT_FEATURES,
    "analog_features": ANALOG_FEATURES,
    "analog_descriptor_features": ANALOG_DESCRIPTOR_FEATURES,
    "local_reference_rmse": LOCAL_REFERENCE_RMSE,
    "public_reference_rmse": PUBLIC_REFERENCE_RMSE,
}
(ARTIFACT_LOGS / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

print(f"wrote {summary_path}")
print(f"wrote {report_path}")
print(f"wrote run bundle {ARTIFACT_LOGS}")

wrote C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\tables\model__typewell_alignment_feature_summary__2026-06-23.csv
wrote C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\reports\model__typewell_alignment_feature_screen__2026-06-23.md
wrote run bundle C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\logs\runs\2026-06-23__typewell_alignment_feature_screen


## 9. Conclusion

In [9]:
"Print the final decision summary."

print(f"best_candidate={best_candidate}")
print(f"sample_baseline_rmse={baseline_row['rmse_mean']:.4f}")
print(f"best_rmse={best_row['rmse_mean']:.4f}")
print(f"delta_vs_sample_baseline={best_row['rmse_delta_vs_sample_baseline']:.4f}")
print(f"promotion_gate_cleared={best_row['rmse_delta_vs_sample_baseline'] <= -0.05}")

best_candidate=analog_direct_s010
sample_baseline_rmse=14.7385
best_rmse=14.6966
delta_vs_sample_baseline=-0.0419
promotion_gate_cleared=False
